In [ ]:
# Author: M. Riley Owens (GitHub: mrileyowens)


In [ ]:
import sys

import os
import glob

import h5py

import numpy as np

from astropy.io import fits
import astropy.units as u
from astropy.coordinates import SkyCoord

from grizli import utils

import matplotlib.pyplot as plt
from matplotlib.offsetbox import AnchoredText

sys.path.append(os.path.abspath('..'))

from mrileyowens.stats import weighted_quantile

In [ ]:
# Set common directories
home = os.getcwd()
data = f'{home}/data'
figs = f'{home}/figs'
results = f'{home}/results'

def select():

    '''
    Plot the SFHs of the 2CSFH BEAGLE models
    '''

    # Set common directories
    #home = os.getcwd()
    #data = f'{home}/data'
    #figs = f'{home}/figs'
    #results = f'{home}/results'

    files = glob.glob(f'{results}/ew/e24_f775w_dropouts_2csfh_no_lya_ews_[!m_uv_e24]*.h5')

    hdul = fits.open(f'{data}/JADES_z6to9LBGcatalog_Endsley2024_f775w_dropouts.fits')

    # Make an empty list to contain the indices and coordinates of selected young, weak emission line sources
    idx_e24, coords_e24 = [], []

    # For each BEAGLE fit results file
    for i, file in enumerate(files):

        with h5py.File(file, 'r') as f:

            for j, id in enumerate(list(f.keys())):

                probs = f[id]['probabilities'][:]

                ews_o_iii, ews_h_alpha, ews_h_beta = f[id]['o_iii_ews'][:], f[id]['h_alpha_ews'][:], f[id]['h_beta_ews'][:]

                p84_h_alpha = weighted_quantile(ews_h_alpha, probs, 0.84)
                p84_o_iii_h_beta = weighted_quantile(ews_o_iii + ews_h_beta, probs, 0.84)

                if p84_h_alpha < 800 and p84_o_iii_h_beta < 800:

                    idx = np.where(hdul[1].data['ID'] == id)

                    f150w = (hdul[1].data['NRC_F150W'] * u.nJy).to(u.ABmag)[idx]
                    f200w = (hdul[1].data['NRC_F200W'] * u.nJy).to(u.ABmag)[idx]
                    f277w = (hdul[1].data['NRC_F277W'] * u.nJy).to(u.ABmag)[idx]

                    color = (f200w - f277w) - (f150w - f200w)

                    if color < 0.3:

                        # Add the object's coordinates to the list of E24 young, weak emission line sources
                        idx_e24.append(idx[0][0])
                        coords_e24.append([hdul[1].data['RA'][idx][0], hdul[1].data['DEC'][idx][0]])

    # Convert the coordinate list to a NumPy array
    idx_e24, coords_e24 = np.array(idx_e24, dtype=np.int64), np.array(coords_e24, dtype=np.float64)

    ids = hdul[1].data['ID'][idx_e24]

    np.savetxt(f'{results}/e24_young_weak_emission_line_ids.txt', ids, fmt='%s')

def f200w():

    '''
    Plot the F200W distribution of the young, weak emission line sources
    '''

    # Get the E24 IDs of the young, weak emission line sources
    ids = np.loadtxt(f'{results}/e24_young_weak_emission_line_ids.txt', dtype=str)

    # Open the HDU list of the E24 catalog
    hdul = fits.open(f'{data}/JADES_z6to9LBGcatalog_Endsley2024_f775w_dropouts.fits')

    # Get the indices of the young, weak emission line sources in the E24 catalog
    idx = np.array([np.where(hdul[1].data['ID'] == id)[0][0] for id in ids], dtype=np.int64)

    # Get the F200W photometry of the young, weak emission line sources
    f200w = (hdul[1].data['NRC_F200W'][idx] * u.nJy).to(u.ABmag)

    # Make a new figure to plot the F200W distribution of the young, weak emission line sources
    fig, ax = plt.subplots()

    # Plot the F200W histogram
    ax.hist(f200w.value, bins=20)

    # Label the axes
    ax.set_xlabel('F200W (AB mag.) (E24)')
    ax.set_ylabel('Count')

    # Save the figure
    fig.savefig(f'{figs}/e24_young_weak_emission_line_f200w.png', bbox_inches='tight', dpi=200)

    plt.close('all')

def download_dja_spectra():

    # Get the E24 IDs of the young, weak emission line sources
    ids_e24 = np.loadtxt(f'{results}/e24_young_weak_emission_line_ids.txt', dtype=str)

    # Open the HDU list of the E24 catalog
    hdul_e24 = fits.open(f'{data}/JADES_z6to9LBGcatalog_Endsley2024_f775w_dropouts.fits')

    # Get the indices of the young, weak emission line sources in the E24 catalog
    idx_e24 = np.array([np.where(hdul_e24[1].data['ID'] == id)[0][0] for id in ids_e24], dtype=np.int64)

    # Convert the coordinate list to a SkyCoord object, to match against the DJA catalog
    coords_e24 = SkyCoord(ra=coords_e24[:,0] * u.deg, dec=coords_e24[:,1] * u.deg)

    # -----------------------------------------------
    # Get the DJA spectroscopic catalog's coordinates
    # -----------------------------------------------

    # Get the roots, file names, and coordinates of all the DJA spectroscopic targets
    roots, files, ra, dec = np.loadtxt(f'{data}/csv.csv', dtype=str, delimiter=',', skiprows=1, usecols=(0,1,2,3), unpack=True)

    # Drop the quotation marks around each entry
    roots, files, ra, dec = np.char.replace(roots, '"', ''), np.char.replace(files, '"', ''), np.char.replace(ra, '"', ''), np.char.replace(dec, '"', '')

    # Convert the coordinate arrays to floats and imbue them with units
    ra_deg, dec_deg = ra.astype(np.float64) * u.deg, dec.astype(np.float64) * u.deg

    # Assemble the catalog's coordinates as a SkyCoord object
    coords_dja = SkyCoord(ra=ra_deg, dec=dec_deg)

    # Find the best matches of the E24 young, weak emission line objects in the spectroscopic DJA catalog
    idx, d2d, _ = coords_e24.match_to_catalog_sky(coords_dja)

    # Set the maximum angular separation between two objects to qualify as a coordinate match
    max_sep = 0.1 * u.arcsec

    # Make a mask of the pairs that are coordinate matches
    mask = d2d < max_sep

    print(table['ra'][idx[mask]], table['dec'][idx[mask]])

In [ ]:
select()

In [ ]:
f200w()